In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

BASE_DIR = "/content/drive/MyDrive/Agent"

MODEL_DIR = os.path.join(BASE_DIR, "Models")

RESULTS_DIR = os.path.join(BASE_DIR, "Results")

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "URL_Analysis_XGBoost.pkl"
)

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Model path:", MODEL_PATH)

Model path: /content/drive/MyDrive/Agent/Models/URL_Analysis_XGBoost.pkl


In [3]:
!pip install -q xgboost pydantic

In [4]:
import pandas as pd
import numpy as np
import joblib
import json
import ipaddress

from urllib.parse import urlparse

from pydantic import BaseModel, Field
from typing import Literal

In [5]:
loaded_model = joblib.load(MODEL_PATH)

print("XGBoost model loaded successfully!")
print("Model classes:", loaded_model.classes_)

XGBoost model loaded successfully!
Model classes: [0 1]


In [6]:
def extract_url_features(url):

    url = str(url).strip()

    parsed = urlparse(
        url if "://" in url else "//" + url
    )

    hostname = parsed.hostname or ""
    path = parsed.path or ""
    query = parsed.query or ""

    features = {}

    features["url_length"] = len(url)
    features["hostname_length"] = len(hostname)
    features["path_length"] = len(path)
    features["query_length"] = len(query)

    features["num_dots"] = url.count(".")
    features["num_hyphens"] = url.count("-")
    features["num_underscores"] = url.count("_")
    features["num_slashes"] = url.count("/")
    features["num_questionmarks"] = url.count("?")
    features["num_equal"] = url.count("=")
    features["num_at"] = url.count("@")
    features["num_ampersands"] = url.count("&")
    features["num_percent"] = url.count("%")

    features["num_digits"] = sum(c.isdigit() for c in url)
    features["num_letters"] = sum(c.isalpha() for c in url)

    features["digit_ratio"] = (
        features["num_digits"] / max(len(url), 1)
    )

    features["https"] = int(url.lower().startswith("https://"))
    features["http"] = int(url.lower().startswith("http://"))

    features["has_ip"] = 0

    try:
        ipaddress.ip_address(hostname)
        features["has_ip"] = 1
    except ValueError:
        pass

    features["num_subdomains"] = max(
        hostname.count(".") - 1, 0
    )

    suspicious_words = [
        "login", "verify", "account", "secure",
        "update", "banking", "password", "confirm",
        "free", "bonus", "wallet", "signin",
        "paypal", "malware", "download"
    ]

    features["suspicious_word_count"] = sum(
        word in url.lower()
        for word in suspicious_words
    )

    features["has_www"] = int(
        "www." in hostname.lower()
    )

    features["has_shortening_service"] = int(
        any(domain in hostname.lower() for domain in [
            "bit.ly", "tinyurl.com", "t.co",
            "goo.gl", "ow.ly", "is.gd"
        ])
    )

    features["special_char_count"] = sum(
        not c.isalnum() for c in url
    )

    return features

In [7]:
class AgentResult(BaseModel):

    agent: str

    status: Literal["completed", "unavailable"]

    classification: Literal[
        "legitimate",
        "suspicious",
        "malicious"
    ]

    risk_score: int | None = Field(
        default=None,
        ge=0,
        le=100
    )

    indicators: list[str] = Field(
        default_factory=list
    )

    explanation: str

    evidence: list[str] = Field(
        default_factory=list
    )

In [8]:
def generate_agent_result(url):

    url = str(url).strip()

    if not url:

        return AgentResult(
            agent="URL_Analysis_Agent",
            status="unavailable",
            classification="suspicious",
            risk_score=None,
            indicators=[],
            explanation="Empty URL. Analysis could not be completed.",
            evidence=[]
        )

    # Extract features
    features = pd.DataFrame([
        extract_url_features(url)
    ])

    # Ensure exact training feature order
    features = features[loaded_model.feature_names_in_]

    # Get probabilities
    probabilities = loaded_model.predict_proba(features)[0]

    # Identify probability positions using class labels
    class_positions = {
        int(label): index
        for index, label in enumerate(loaded_model.classes_)
    }

    legitimate_probability = float(
        probabilities[class_positions[0]]
    )

    malicious_probability = float(
        probabilities[class_positions[1]]
    )

    # Classification
    if malicious_probability >= 0.70:

        classification = "malicious"

    elif malicious_probability <= 0.30:

        classification = "legitimate"

    else:

        classification = "suspicious"

    # URL indicators
    feature_values = features.iloc[0]

    indicators = []

    if feature_values["has_ip"] == 1:
        indicators.append("URL uses an IP address.")

    if feature_values["suspicious_word_count"] > 0:
        indicators.append("Potentially suspicious keywords detected.")

    if feature_values["has_shortening_service"] == 1:
        indicators.append("URL shortening service detected.")

    if feature_values["num_subdomains"] >= 3:
        indicators.append("Multiple subdomains detected.")

    if feature_values["num_at"] > 0:
        indicators.append("@ symbol detected in URL.")

    if feature_values["num_digits"] > 10:
        indicators.append("High number of digits detected.")

    if feature_values["num_hyphens"] >= 3:
        indicators.append("Multiple hyphens detected.")

    if not indicators:
        indicators.append("No predefined URL indicators detected.")

    # Explanation
    explanation = (
        f"XGBoost classified this URL as {classification}. "
        f"The predicted malicious probability is "
        f"{malicious_probability * 100:.2f}%. "
        "This prediction is based on learned URL characteristics."
    )

    # Evidence
    evidence = [
        f"URL: {url}",
        f"Malicious probability: {malicious_probability * 100:.2f}%",
        f"Legitimate probability: {legitimate_probability * 100:.2f}%",
        "Model: XGBoost",
        "Feature extraction: Numerical URL lexical features"
    ]

    return AgentResult(
        agent="URL_Analysis_Agent",
        status="completed",
        classification=classification,
        risk_score=None,
        indicators=indicators,
        explanation=explanation,
        evidence=evidence
    )

In [9]:
test_url = "https://www.google.com"

result = generate_agent_result(test_url)

print(result.model_dump_json(indent=4))

{
    "agent": "URL_Analysis_Agent",
    "status": "completed",
    "classification": "legitimate",
    "risk_score": null,
    "indicators": [
        "No predefined URL indicators detected."
    ],
    "explanation": "XGBoost classified this URL as legitimate. The predicted malicious probability is 0.69%. This prediction is based on learned URL characteristics.",
    "evidence": [
        "URL: https://www.google.com",
        "Malicious probability: 0.69%",
        "Legitimate probability: 99.31%",
        "Model: XGBoost",
        "Feature extraction: Numerical URL lexical features"
    ]
}


In [10]:
test_urls = [
    "https://www.google.com",
    "https://www.facebook.com",
    "http://192.168.1.1/login",
    "http://example-test-url.com/verify-account"
]

for url in test_urls:

    result = generate_agent_result(url)

    print("\nURL:", url)

    print(result.model_dump_json(indent=4))

    print("-" * 60)


URL: https://www.google.com
{
    "agent": "URL_Analysis_Agent",
    "status": "completed",
    "classification": "legitimate",
    "risk_score": null,
    "indicators": [
        "No predefined URL indicators detected."
    ],
    "explanation": "XGBoost classified this URL as legitimate. The predicted malicious probability is 0.69%. This prediction is based on learned URL characteristics.",
    "evidence": [
        "URL: https://www.google.com",
        "Malicious probability: 0.69%",
        "Legitimate probability: 99.31%",
        "Model: XGBoost",
        "Feature extraction: Numerical URL lexical features"
    ]
}
------------------------------------------------------------

URL: https://www.facebook.com
{
    "agent": "URL_Analysis_Agent",
    "status": "completed",
    "classification": "legitimate",
    "risk_score": null,
    "indicators": [
        "No predefined URL indicators detected."
    ],
    "explanation": "XGBoost classified this URL as legitimate. The predicted 